In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

I0000 00:00:1773340425.292689   37338 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773340425.293303   37338 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773340425.351619   37338 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773340426.856959   37338 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [2]:
#load imgclf_resnet50_cifar10_v1
imgclf_resnet50_model = keras.models.load_model("imgclf_resnet50_cifar10_v1.keras")

E0000 00:00:1773340428.187436   37338 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
#STILL TO GET
#load imgclf_effnetb0_cifar10_v1.keras
#imgclf_effnetb0_model = keras.models.load_model("imgclf_effnetb0_cifar10_v1.keras")

In [4]:
class_names = {
    0:"airplane", 
    1:"automobile", 
    2:"bird", 
    3:"cat", 
    4:"deer",
    5:"dog", 
    6:"frog", 
    7:"horse", 
    8:"ship", 
    9:"truck"
}

# Image Preprocessing

## Upload image

In [16]:
img = cv2.imread('cat.jpeg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

## resize and padding

In [17]:
height, width, ch = img.shape

w_longest =  width > height
max_side = np.max([height, width])
min_side = np.min([height, width])

In [18]:
resized_max_side = 32
resized_min_side = int(32 * min_side / max_side) if height != width else 32

In [19]:
dsize = (resized_max_side, resized_min_side) if w_longest else (resized_min_side, resized_max_side)
img_resized = cv2.resize(img, dsize, interpolation=cv2.INTER_AREA)

In [20]:
padd_tuple = (int(np.ceil((32 - resized_min_side)/2)), int((32 - resized_min_side)/2), 0, 0)

if w_longest:
  top, bottom, left, right = padd_tuple
else:
  left, right, top, bottom = padd_tuple

In [21]:
img_prepr = cv2.copyMakeBorder(img_resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)

## Model Preprocessing

In [22]:
img_prepr_resnet = resnet_preprocess(img_prepr)
img_prepr_resnet = np.array([img_prepr_resnet])

In [23]:
img_prepr_effnet = efficientnet_preprocess(img_prepr)
img_prepr_effnet = np.array([img_prepr_effnet])

# Prediction

In [24]:
#prediction with imgclf_resnet50_model
class_prob_resnet = imgclf_resnet50_model.predict(img_prepr_resnet)
class_pred_resnet = np.argmax(class_prob_resnet, axis=1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step


In [25]:
#prediction with imgclf_effnetb0_model
#class_prob_effnet = imgclf_effnetb0_model.predict(img_prepr_effnet)
#class_pred_effnet = np.argmax(class_prob_effnet, axis=1)

In [26]:
print("imgclf_resnet50_cifar10_v1:", class_names[int(class_pred_resnet[0])])
#print("imgclf_effnetb0_cifar10_v1:", class_names[int(class_pred_effnet[0])])

imgclf_resnet50_cifar10_v1: cat
